# Quantum Belief Update Circuit

Demonstrates the quantum circuit for POMDP belief update (arXiv:2507.18606).

Circuit structure:
```
|0>_{s_t}   ---[ U(b) ]---[ U_1 ]---[     ]---[     ]---[ G^k(o) ]--- Measure
|0>_{a_t}   ---[ U(a) ]---[     ]---[     ]---[     ]---[        ]---
|0>_{s_t+1} ---[      ]---[     ]---[ U_2 ]---[     ]---[        ]--- Measure
|0>_{o_t+1} ---[      ]---[     ]---[     ]---[     ]---[        ]--- Measure
|0>_{r_t+1} ---[      ]---[     ]---[     ]---[ U_3 ]---[        ]--- Measure
```

In [ ]:
from quantum_pomdp.scenarios.tiger_problem import create_tiger_pomdp
from quantum_pomdp.models.belief_state import BeliefState
from quantum_pomdp.quantum_circuits.belief_update import (
    BeliefUpdateCircuitConfig,
    QuantumBeliefUpdateCircuit,
)
from quantum_pomdp.quantum_circuits.register_map import POMDPRegisterMap

model = create_tiger_pomdp()
belief = BeliefState.uniform(2)
print(f"Tiger POMDP: {model.num_states}S, {model.num_actions}A, {model.num_observations}O")

In [ ]:
# Register allocation
reg = POMDPRegisterMap.from_dimensions(
    state_qubits=model.state_qubits,
    action_qubits=model.action_qubits,
    observation_qubits=model.observation_qubits,
)
print(f"State register: {reg.state_current.size} qubits")
print(f"Action register: {reg.action.size} qubits")
print(f"Next state register: {reg.state_next.size} qubits")
print(f"Observation register: {reg.observation.size} qubits")
print(f"Reward register: {reg.reward.size} qubits")
print(f"Total qubits: {reg.total_qubits}")

In [ ]:
# Build circuit without amplitude amplification
config = BeliefUpdateCircuitConfig(
    use_amplitude_amplification=False,
    include_reward_register=False,
)
builder = QuantumBeliefUpdateCircuit(model, config)
circuit = builder.build_without_aa(belief, action=0)  # Listen

info = builder.get_circuit_info(belief, action=0)
print(f"Circuit info: {info}")
print(f"\nCircuit depth: {circuit.depth()}")
print(f"Circuit size: {circuit.size()}")

In [ ]:
# Build full circuit with amplitude amplification
config_aa = BeliefUpdateCircuitConfig(
    use_amplitude_amplification=True,
    aa_iterations=1,
    include_reward_register=False,
)
builder_aa = QuantumBeliefUpdateCircuit(model, config_aa)
circuit_aa = builder_aa.build(belief, action=0, observation=0)

print(f"With AA - depth: {circuit_aa.depth()}, size: {circuit_aa.size()}")
print(f"Quantum advantage: quadratic speedup via Grover AA")